<a href="https://colab.research.google.com/github/sathursiyakrishnamoorthy/Data-Science-Project-Hotel-A/blob/Model-choosing-phase/choosing_the_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# PREPROCESSING
def preprocess_for_all(df):
    df = df.copy()
    if 'Reservation_Status' in df.columns:
        df['Reservation_Status'] = df['Reservation_Status'].str.strip().str.lower()
        status_map = {'check-in': 1, 'check-out': 1, 'canceled': 2, 'no-show': 3}
        df['Reservation_Status'] = df['Reservation_Status'].map(status_map)
    for col in ['Expected_checkin', 'Expected_checkout', 'Booking_date']:
        df[col] = pd.to_datetime(df[col], format='mixed')
    df['LeadTime'] = (df['Expected_checkin'] - df['Booking_date']).dt.days
    df['StayDuration'] = (df['Expected_checkout'] - df['Expected_checkin']).dt.days
    le = LabelEncoder()
    categorical_cols = ['Gender', 'Ethnicity', 'Educational_Level', 'Income', 'Country_region',
                        'Hotel_Type', 'Meal_Type', 'Deposit_type', 'Booking_channel']
    for col in categorical_cols:
        df[col] = le.fit_transform(df[col].astype(str))
    binary_cols = ['Visted_Previously', 'Previous_Cancellations', 'Required_Car_Parking', 'Use_Promotion']
    for col in binary_cols:
        df[col] = df[col].map({'Yes': 1, 'No': 0})
    cols_to_drop = ['Reservation-id', 'Expected_checkin', 'Expected_checkout', 'Booking_date']
    df = df.drop(columns=cols_to_drop)
    return df.fillna(0)

# Loading data and preparing features

train_df = preprocess_for_all(pd.read_csv('/content/drive/MyDrive/Data Science/Hotel-A-train (1).csv'))
val_df = preprocess_for_all(pd.read_csv('/content/drive/MyDrive/Data Science/Hotel-A-validation (1).csv'))

X_train, y_train = train_df.drop('Reservation_Status', axis=1), train_df['Reservation_Status']
X_val, y_val = val_df.drop('Reservation_Status', axis=1), val_df['Reservation_Status']

#  FEATURE SCALING
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Training SVM Model
print("Training Support Vector Machine (SVM)...")

svm_model = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_acc = accuracy_score(y_val, svm_model.predict(X_val_scaled))

# Training kNN Model
print("Training k-Nearest Neighbors (kNN)...")
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)
knn_acc = accuracy_score(y_val, knn_model.predict(X_val_scaled))

# FINAL COMPARISON
print("PERFORMANCE SUMMARY")
print(f"SVM Accuracy: {svm_acc:.4f}")
print(f"kNN Accuracy: {knn_acc:.4f}")

Training Support Vector Machine (SVM)...
Training k-Nearest Neighbors (kNN)...
PERFORMANCE SUMMARY
SVM Accuracy: 0.3630
kNN Accuracy: 0.5766


In [7]:
from sklearn.metrics import classification_report

# Evaluation for kNN
print("--- kNN CLASSIFICATION REPORT ---")

y_pred_knn = knn_model.predict(X_val_scaled)
print(classification_report(y_val, y_pred_knn, target_names=['Check-In', 'Canceled', 'No-Show']))



--- kNN CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

    Check-In       0.59      0.96      0.73      1610
    Canceled       0.34      0.05      0.08       741
     No-Show       0.19      0.01      0.01       398

    accuracy                           0.58      2749
   macro avg       0.37      0.34      0.28      2749
weighted avg       0.46      0.58      0.45      2749



In [8]:
# Evaluation for SVM Model
print("\n--- SVM CLASSIFICATION REPORT ---")

y_pred_svm = svm_model.predict(X_val_scaled)
print(classification_report(y_val, y_pred_svm, target_names=['Check-In', 'Canceled', 'No-Show']))


--- SVM CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

    Check-In       0.62      0.36      0.46      1610
    Canceled       0.29      0.38      0.33       741
     No-Show       0.16      0.32      0.21       398

    accuracy                           0.36      2749
   macro avg       0.36      0.36      0.33      2749
weighted avg       0.47      0.36      0.39      2749



In [9]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score

# Initialising the Naive Bayes Model
nb_model = GaussianNB()

# Training the model

nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_val)

 # Evaluatation for NB
print(f"Naive Bayes Accuracy: {accuracy_score(y_val, y_pred_nb):.4f}")
print("\nNaive Bayes Classification Report:")
print(classification_report(y_val, y_pred_nb, target_names=['Check-In/Out', 'Canceled', 'No-Show']))

Naive Bayes Accuracy: 0.5857

Naive Bayes Classification Report:
              precision    recall  f1-score   support

Check-In/Out       0.59      1.00      0.74      1610
    Canceled       0.00      0.00      0.00       741
     No-Show       0.00      0.00      0.00       398

    accuracy                           0.59      2749
   macro avg       0.20      0.33      0.25      2749
weighted avg       0.34      0.59      0.43      2749



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
